In [6]:
# Project: NHANES Harmonized Dataset Verification
# Dataset: Labs
# Years: 2003–2014
# Purpose: Verify Kaggle harmonized labs data against raw NHANES FASTQX, GLU, HDL, and TCHOL files.
# Source Raw Data: CDC NHANES FASTQX_C-H, GLU_C-H, HDL_C-H, TCHOL_C-H XPT files
# Harmonized Data: Kaggle NHANES harmonized dataset
# Developer: Neha Khairnar
# Kaggle Source: https://www.kaggle.com/datasets/nguyenvy/nhanes-19882018
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "datasets" / "NHANE_Continuous_DemographicsData_Appended_2003_2014_Data_Labs_only"

OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DIR = BASE_DIR / "data" / "datasets" / "NHANE_HARMONIZED" / "archive"

# Update if your Kaggle lab file has different name
kaggle_file = KAGGLE_DIR / "response_clean.csv"

# =================================================================================================
# Step 2: Raw NHANES LAB Files: 2003–2014
# =================================================================================================

lab_files = {
    "2003-2004": {
        "fasting": "PH_C.XPT",
        "glucose": "L10AM_C.XPT",
        "hdl": "L13_C.XPT",
        "total_cholesterol": "L13AM_C.XPT",
    },
    "2005-2006": {
        "fasting": "FASTQX_D.XPT",
        "glucose": "GLU_D.XPT",
        "hdl": "HDL_D.XPT",
        "total_cholesterol": "TCHOL_D.XPT",
    },
    "2007-2008": {
        "fasting": "FASTQX_E.XPT",
        "glucose": "GLU_E.XPT",
        "hdl": "HDL_E.XPT",
        "total_cholesterol": "TCHOL_E.XPT",
    },
    "2009-2010": {
        "fasting": "FASTQX_F.XPT",
        "glucose": "GLU_F.XPT",
        "hdl": "HDL_F.XPT",
        "total_cholesterol": "TCHOL_F.XPT",
    },
    "2011-2012": {
        "fasting": "FASTQX_G.XPT",
        "glucose": "GLU_G.XPT",
        "hdl": "HDL_G.XPT",
        "total_cholesterol": "TCHOL_G.XPT",
    },
    "2013-2014": {
        "fasting": "FASTQX_H.XPT",
        "glucose": "GLU_H.XPT",
        "hdl": "HDL_H.XPT",
        "total_cholesterol": "TCHOL_H.XPT",
    },
}

cycle_map = {
    "2003-2004": 3,
    "2005-2006": 4,
    "2007-2008": 5,
    "2009-2010": 6,
    "2011-2012": 7,
    "2013-2014": 8,
}

# =================================================================================================
# Step 3: Load Raw NHANES LAB Files
# =================================================================================================

raw_labs_list = []

for cycle, files in lab_files.items():
    cycle_dfs = []

    for lab_name, filename in files.items():
        file_path = DATA_DIR / filename

        print(f"Loading {cycle} | {lab_name} | {filename}")

        df = pd.read_sas(file_path, format="xport")
        df.columns = df.columns.str.upper()

        df["CYCLE"] = cycle
        df["SDDSRVYR"] = cycle_map[cycle]
        df["LAB_DOMAIN"] = lab_name

        cycle_dfs.append(df)

    cycle_labs = cycle_dfs[0]

    for df in cycle_dfs[1:]:
        df = df.drop(columns=["LAB_DOMAIN"], errors="ignore")
        cycle_labs = cycle_labs.merge(
            df,
            on=["SEQN", "CYCLE", "SDDSRVYR"],
            how="outer"
        )

    raw_labs_list.append(cycle_labs)

raw_labs = pd.concat(raw_labs_list, ignore_index=True)

print("Raw labs shape:", raw_labs.shape)
display(raw_labs.head())

# =================================================================================================
# Step 4: Load Patel Kaggle Harmonized Labs File
# =================================================================================================

kaggle_labs = pd.read_csv(kaggle_file)
kaggle_labs.columns = kaggle_labs.columns.str.upper()

Kaggle_labs_2003_2014 = kaggle_labs[
    kaggle_labs["SDDSRVYR"].isin([3, 4, 5, 6, 7, 8])
].copy()

print("Kaggle labs total shape:", kaggle_labs.shape)
print("Kaggle labs 2003-2014 shape:", Kaggle_labs_2003_2014.shape)

display(Kaggle_labs_2003_2014.head())

# =================================================================================================
# Step 5: Inspect Columns
# =================================================================================================

print("\nInspecting columns for RAW NHANES Labs vs Harmonized Kaggle Labs")
print("=" * 80)

print("\nRAW NHANES LAB columns:")
print(raw_labs.columns.tolist())

print("\nKAGGLE LAB columns:")
print(Kaggle_labs_2003_2014.columns.tolist())

print("=" * 80)

print("\nRow counts:")
print("Raw NHANES labs rows:", len(raw_labs))
print("Kaggle labs total rows:", len(kaggle_labs))
print("Kaggle labs rows for 2003-2014:", len(Kaggle_labs_2003_2014))

# =================================================================================================
# Step 6: Compare Row Counts by Cycle
# =================================================================================================

raw_cycle_counts = raw_labs.groupby("SDDSRVYR")["SEQN"].nunique().reset_index()
raw_cycle_counts.rename(columns={"SEQN": "raw_unique_seqn"}, inplace=True)

kaggle_cycle_counts = Kaggle_labs_2003_2014.groupby("SDDSRVYR")["SEQN"].nunique().reset_index()
kaggle_cycle_counts.rename(columns={"SEQN": "kaggle_unique_seqn"}, inplace=True)

cycle_count_compare = raw_cycle_counts.merge(
    kaggle_cycle_counts,
    on="SDDSRVYR",
    how="outer"
)

cycle_count_compare["difference_raw_minus_kaggle"] = (
    cycle_count_compare["raw_unique_seqn"] - cycle_count_compare["kaggle_unique_seqn"]
)

print("\nCycle Count Comparison:")
display(cycle_count_compare)

# =================================================================================================
# Step 7: Compare SEQN Values
# =================================================================================================

raw_seqn = set(raw_labs["SEQN"])
kaggle_seqn = set(Kaggle_labs_2003_2014["SEQN"])

missing_in_kaggle_seqn = sorted(raw_seqn - kaggle_seqn)
extra_in_kaggle_seqn = sorted(kaggle_seqn - raw_seqn)

print("\nSEQN comparison:")
print("Missing SEQN in Kaggle:", len(missing_in_kaggle_seqn))
print("Extra SEQN in Kaggle:", len(extra_in_kaggle_seqn))

# Participants existing in both datasets
verified_labs = raw_labs.merge(
    Kaggle_labs_2003_2014,
    on="SEQN",
    how="inner",
    suffixes=("_RAW", "_KAGGLE")
)

print("Verified lab participants:", len(verified_labs))

# =================================================================================================
# Step 8: Search Lab Variables in Kaggle Dataset
# =================================================================================================

search_terms = [
    "GLU", "LBXGLU",
    "HDL", "HDD", "LBDHDD", "LBXHDD",
    "CHOL", "TCHOL", "LBXTC",
    "FAST", "PHAFST"
]

matched_kaggle_lab_cols = [
    col for col in Kaggle_labs_2003_2014.columns
    if any(term in col.upper() for term in search_terms)
]

print("\nPossible lab-related variables found in Kaggle:")
for col in matched_kaggle_lab_cols:
    print(col)

# =================================================================================================
# Step 9: Check Specific Expected Raw Lab Variables
# =================================================================================================

expected_raw_lab_vars = [
    "SEQN",
    "SDDSRVYR",
    "LBXGLU",
    "LBXTC",
    "LBDHDD",
    "LBXHDD",
    "PHAFSTHR",
    "PHAFSTMN"
]

specific_check = []

for var in expected_raw_lab_vars:
    specific_check.append({
        "variable": var,
        "exists_in_raw": var in raw_labs.columns,
        "exists_in_kaggle": var in Kaggle_labs_2003_2014.columns
    })

specific_check_df = pd.DataFrame(specific_check)

print("\nSpecific lab variable check:")
display(specific_check_df)

# =================================================================================================
# Step 10: Compare Every Column
# =================================================================================================

raw_cols = set(raw_labs.columns)
kaggle_cols = set(Kaggle_labs_2003_2014.columns)

common_cols = sorted(raw_cols.intersection(kaggle_cols))
missing_in_kaggle = sorted(raw_cols - kaggle_cols)
extra_in_kaggle = sorted(kaggle_cols - raw_cols)

print("=" * 80)
print("RAW NHANES LAB COLUMNS:", len(raw_cols))

print("=" * 80)
print("PATEL/KAGGLE LAB COLUMNS:", len(kaggle_cols))

print("=" * 80)
print("COMMON COLUMNS:", len(common_cols))

print("=" * 80)
print("MISSING IN KAGGLE:", len(missing_in_kaggle))

print("=" * 80)
print("EXTRA IN KAGGLE:", len(extra_in_kaggle))

print("=" * 80)
print("\nCOMMON COLUMNS")
for c in common_cols:
    print(c)

print("=" * 80)
print("\nMISSING IN KAGGLE")
for c in missing_in_kaggle:
    print(c)

print("=" * 80)
print("\nEXTRA IN KAGGLE")
for c in extra_in_kaggle:
    print(c)

# =================================================================================================
# Step 11: Generate Column Verification Report
# =================================================================================================

report = pd.DataFrame({
    "column_name": sorted(raw_cols.union(kaggle_cols))
})

report["exists_in_raw"] = report["column_name"].isin(raw_cols)
report["exists_in_kaggle"] = report["column_name"].isin(kaggle_cols)

report["status"] = report.apply(
    lambda x: "YES" if x["exists_in_raw"] and x["exists_in_kaggle"] else "NO",
    axis=1
)

report["category"] = report.apply(
    lambda x:
        "COMMON"
        if x["exists_in_raw"] and x["exists_in_kaggle"]
        else "MISSING_IN_KAGGLE"
        if x["exists_in_raw"] and not x["exists_in_kaggle"]
        else "EXTRA_IN_KAGGLE",
    axis=1
)

display(report)

# =================================================================================================
# Step 12: Final Kaggle Dataset with ONLY Common Raw Lab Columns
# =================================================================================================

kaggle_common_only = Kaggle_labs_2003_2014[common_cols].copy()

print("Kaggle labs dataset with common raw columns only:")
print(kaggle_common_only.shape)

display(kaggle_common_only.head())

# =================================================================================================
# Step 13: Kaggle Dataset with Missing Raw Columns Added for Verification
# =================================================================================================

kaggle_with_missing_raw_cols = Kaggle_labs_2003_2014.copy()

for col in missing_in_kaggle:
    kaggle_with_missing_raw_cols[col] = pd.NA

final_column_order = common_cols + missing_in_kaggle

kaggle_with_missing_raw_cols = kaggle_with_missing_raw_cols[final_column_order].copy()

print("Kaggle labs dataset with missing raw columns added:")
print(kaggle_with_missing_raw_cols.shape)

display(kaggle_with_missing_raw_cols.head())

# =================================================================================================
# Step 14: Export Verification Outputs
# =================================================================================================

report.to_csv(
    OUTPUT_DIR / "NHANES_labs_raw_vs_kaggle_column_verification_report.csv",
    index=False
)

specific_check_df.to_csv(
    OUTPUT_DIR / "NHANES_labs_specific_variable_check.csv",
    index=False
)

cycle_count_compare.to_csv(
    OUTPUT_DIR / "NHANES_labs_cycle_count_comparison.csv",
    index=False
)

pd.DataFrame({
    "missing_seqn_in_kaggle": missing_in_kaggle_seqn
}).to_csv(
    OUTPUT_DIR / "NHANES_labs_missing_seqn_in_kaggle.csv",
    index=False
)

pd.DataFrame({
    "extra_seqn_in_kaggle": extra_in_kaggle_seqn
}).to_csv(
    OUTPUT_DIR / "NHANES_labs_extra_seqn_in_kaggle.csv",
    index=False
)

kaggle_common_only.to_csv(
    OUTPUT_DIR / "NHANES_labs_kaggle_common_raw_columns_only.csv",
    index=False
)

kaggle_with_missing_raw_cols.to_csv(
    OUTPUT_DIR / "NHANES_labs_kaggle_with_missing_raw_columns_added.csv",
    index=False
)

print("Verification outputs saved to:")
print(OUTPUT_DIR)

Loading 2003-2004 | fasting | PH_C.XPT
Loading 2003-2004 | glucose | L10AM_C.XPT
Loading 2003-2004 | hdl | L13_C.XPT
Loading 2003-2004 | total_cholesterol | L13AM_C.XPT
Loading 2005-2006 | fasting | FASTQX_D.XPT
Loading 2005-2006 | glucose | GLU_D.XPT
Loading 2005-2006 | hdl | HDL_D.XPT
Loading 2005-2006 | total_cholesterol | TCHOL_D.XPT
Loading 2007-2008 | fasting | FASTQX_E.XPT
Loading 2007-2008 | glucose | GLU_E.XPT
Loading 2007-2008 | hdl | HDL_E.XPT
Loading 2007-2008 | total_cholesterol | TCHOL_E.XPT
Loading 2009-2010 | fasting | FASTQX_F.XPT
Loading 2009-2010 | glucose | GLU_F.XPT
Loading 2009-2010 | hdl | HDL_F.XPT
Loading 2009-2010 | total_cholesterol | TCHOL_F.XPT
Loading 2011-2012 | fasting | FASTQX_G.XPT
Loading 2011-2012 | glucose | GLU_G.XPT
Loading 2011-2012 | hdl | HDL_G.XPT
Loading 2011-2012 | total_cholesterol | TCHOL_G.XPT
Loading 2013-2014 | fasting | FASTQX_H.XPT
Loading 2013-2014 | glucose | GLU_H.XPT
Loading 2013-2014 | hdl | HDL_H.XPT
Loading 2013-2014 | total_ch

,SEQN,PHQ020,PHACOFHR,PHACOFMN,PHQ030,PHAALCHR,PHAALCMN,PHQ040,PHAGUMHR,PHAGUMMN,...,LBXTR,LBDTRSI,LBDLDL,LBDLDLSI,PHAFSTHR_x,PHAFSTMN_x,WTSAF2YR,PHAFSTHR_y,PHAFSTMN_y,LBDHDD
0,21005.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,...,102.0,1.152,125.0,3.233,NaN,NaN,NaN,NaN,NaN,NaN
1,21006.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,...,71.0,0.802,92.0,2.379,NaN,NaN,NaN,NaN,NaN,NaN
2,21007.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21008.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21009.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


C:\Users\NVK00\AppData\Local\Temp\ipykernel_19712\3536152560.py:119: DtypeWarning: Columns (0: PHPBEST2, 1: PHPDRTI2, 2: PHPSNTI2, 3: PHPBEST1, 4: PHPDRTI1, 5: PHPSNTI1, 6: SPPTIME1, 7: SPPTIME2, 8: SPXBQEFF, 9: SPXBQFV1, 10: SPXBQFVC, 11: SPXNQEFF, 12: SPXNQFV1, 13: SPXNQFVC) have mixed types. Specify dtype option on import or set low_memory=False.
  kaggle_labs = pd.read_csv(kaggle_file)


Kaggle labs total shape: (131030, 1038)
Kaggle labs 2003-2014 shape: (60200, 1038)


,SEQN,SEQN_NEW,SDDSRVYR,LBXSATSI2,LBXSATSI,LBXSATSI1,LBDSATLC,LBDSALSI2,LBXSAL2,URXUMA2,...,LBXTFR,LBDSUASI2,LBXSUA2,LBDSUASI,LBDSUASI1,LBXSUA,LBXSUA1,URDNT,URDNT1,URDNT2
51985,21005,C-21005,3,NaN,15.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,404.5,NaN,6.8,NaN,NaN,NaN,NaN
51986,21006,C-21006,3,NaN,29.0,NaN,NaN,NaN,NaN,NaN,...,3.2,NaN,NaN,160.6,NaN,2.7,NaN,NaN,NaN,NaN
51987,21007,C-21007,3,NaN,14.0,NaN,NaN,NaN,NaN,NaN,...,2.6,NaN,NaN,309.3,NaN,5.2,NaN,NaN,NaN,NaN
51988,21008,C-21008,3,NaN,12.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,333.1,NaN,5.6,NaN,NaN,NaN,NaN
51989,21009,C-21009,3,NaN,22.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,404.5,NaN,6.8,NaN,NaN,NaN,NaN


...

Kaggle labs dataset with common raw columns only:
(60200, 17)


,LBDHDD,LBDHDDSI,LBDINSI,LBDLDL,LBDLDLSI,LBDTCSI,LBDTRSI,LBXCPSI,LBXGLU,LBXIN,LBXTC,LBXTR,PHAFSTHR,PHAFSTMN,SDDSRVYR,SEQN,WTSAF2YR
51985,44.0,1.14,67.56,125.0,3.233,4.89,1.152,0.591,87.6,11.26,189.0,102.0,10.0,2.0,3,21005,14084.2
51986,53.0,1.37,68.64,92.0,2.379,4.11,0.802,0.535,75.0,11.44,159.0,71.0,12.0,22.0,3,21006,13081.0
51987,69.0,1.78,NaN,NaN,NaN,5.48,NaN,NaN,NaN,NaN,212.0,NaN,20.0,55.0,3,21007,NaN
51988,66.0,1.71,NaN,NaN,NaN,4.53,NaN,NaN,NaN,NaN,175.0,NaN,4.0,35.0,3,21008,NaN
51989,37.0,0.96,NaN,NaN,NaN,6.57,NaN,NaN,NaN,NaN,254.0,NaN,6.0,15.0,3,21009,NaN


Kaggle labs dataset with missing raw columns added:
(60200, 43)


,LBDHDD,LBDHDDSI,LBDINSI,LBDLDL,LBDLDLSI,LBDTCSI,LBDTRSI,LBXCPSI,LBXGLU,LBXIN,...,PHASUPHR,PHASUPMN,PHDSESN,PHQ020,PHQ030,PHQ040,PHQ050,PHQ060,WTSAF2YR_x,WTSAF2YR_y
51985,44.0,1.14,67.56,125.0,3.233,4.89,1.152,0.591,87.6,11.26,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
51986,53.0,1.37,68.64,92.0,2.379,4.11,0.802,0.535,75.0,11.44,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
51987,69.0,1.78,NaN,NaN,NaN,5.48,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
51988,66.0,1.71,NaN,NaN,NaN,4.53,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
51989,37.0,0.96,NaN,NaN,NaN,6.57,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


Verification outputs saved to:
C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed
